In [ ]:
import helpers.Create_Target_Variable as Create_Target_Variable
import helpers.create_team_tendency_features as create_team_tendency_features
import pandas as pd
import re

In [2]:
# gather data
plays_df = pd.DataFrame()

plays_files = ['2017_plays.csv',
               '2018_plays.csv',
               '2019_plays.csv',
               '2020_plays.csv',
               '2021_plays.csv',
               '2022_plays.csv',
               '2023_plays.csv',
               '2024_plays.csv',
               '2025_plays.csv'
               ]

for file in plays_files:
    temp_df = pd.read_csv('Classify Plays/' + file)
    plays_df = pd.concat([plays_df, temp_df])

In [3]:
# drop rows where PlayStart is null - irrelevant data
plays_df.dropna(subset=['PlayStart'], inplace=True)

# drop sacks and penalty plays - can't classify a play call
plays_df = plays_df[~plays_df['PlayOutcome'].str.contains('sack|penalty', case=False, na=False)]

# remove 'Hall Of Fame' and 'Preseason' games
plays_df = plays_df[~plays_df['Week'].str.contains('Hall Of Fame|Preseason|18|Wild|Conference|Divisional|Super', na=False)]

# remove the 'Week' prefix
plays_df['Week'] = plays_df['Week'].str[-2:].str.strip()
# Convert Week to int
plays_df['Week'] = plays_df['Week'].astype(int)

In [ ]:
# create target variable: play call
plays_df['PlayAttempt'] = plays_df.apply(Create_Target_Variable.classify_play_call, axis=1)

In [ ]:
# drop where play cannot be classified
plays_df.dropna(subset=['PlayAttempt'], inplace=True)

In [6]:
# map team with possesion to abbreviated name
team_name_map = {
    'Arizona Cardinals': 'ARI',
    'Dallas Cowboys': 'DAL',
    'Houston Texans': 'HOU',
    'Carolina Panthers': 'CAR',
    'Minnesota Vikings': 'MIN',
    'Buffalo Bills': 'BUF',
    'Miami Dolphins': 'MIA',
    'Atlanta Falcons': 'ATL',
    'Washington Commanders': 'WAS',
    'Baltimore Ravens': 'BAL',
    'Jacksonville Jaguars': 'JAX',
    'New England Patriots': 'NE',
    'Chicago Bears': 'CHI',
    'Denver Broncos': 'DEN',
    'New Orleans Saints': 'NO',
    'Cleveland Browns': 'CLE',
    'Green Bay Packers': 'GB',
    'Philadelphia Eagles': 'PHI',
    'Pittsburgh Steelers': 'PIT',
    'New York Giants': 'NYG',
    'Tampa Bay Buccaneers': 'TB',
    'Cincinnati Bengals': 'CIN',
    'Kansas City Chiefs': 'KC',
    'San Francisco 49ers': 'SF',
    'New York Jets': 'NYJ',
    'Tennessee Titans': 'TEN',
    'Los Angeles Rams': 'LAR',
    'Las Vegas Raiders': 'LV',
    'Detroit Lions': 'DET',
    'Indianapolis Colts': 'IND',
    'Los Angeles Chargers': 'LAC',
    'Seattle Seahawks': 'SEA'
}

plays_df['TeamWithPossessionShort'] = plays_df['TeamWithPossession'].map(team_name_map)

In [7]:
### extract data from PlayStart field e.g. "3rd & 1 at LAC 1" ###

# split before and after " at "
temp = plays_df['PlayStart'].str.split(' at ', expand=True)
left = temp[0]   # "3rd & 1"
right = temp[1]  # "LAC 1"

# extract Down and YdsTo1stDown
plays_df['Down'] = left.str.extract(r'(\d+)').astype(int)
plays_df['YdsTo1stDown'] = left.str.extract(r'&\s*(\d+)').astype(int)

# extract Territory and YdPosition
plays_df['Territory'] = right.str.extract(r'([A-Z]+)')
plays_df['YdPosition'] = right.str.extract(r'(\d+)').astype(int)

In [8]:
# convert JAC to JAX and OAK to LV in Territory field
plays_df['Territory'] = plays_df['Territory'].replace({
    'JAC': 'JAX',
    'OAK': 'LV'
})

In [9]:
# calculate yards to enzone
def get_yds_to_endzone(row):
    if row['YdPosition'] == 50:
        return 50
    elif row['Territory'] == row['TeamWithPossessionShort']:
        return row['YdPosition'] + 50
    else:
        return row['YdPosition']

plays_df['YdsToEndzone'] = plays_df.apply(get_yds_to_endzone, axis=1)

In [15]:
# Calculate global means for fallback (using all seasons data)
global_means = create_team_tendency_features.calculate_season_means(plays_df, season=None)

LEAGUE AVERAGES FOR ALL SEASONS
Run         : 43.87%
Short_Pass  : 51.27%
Long_Pass   :  4.86%



In [22]:
# Add team tendency features for each season
all_seasons_with_features = []

for year in range(2017, 2026):  # 2017-2025
    print(f"\nProcessing {year}...")
    
    # Get current season data
    df_season = plays_df[plays_df['Season'] == year].copy()
    
    # Get previous season data (None for 2017)
    if year == 2017:
        df_previous = None
        print(f"  Using global means (first season)")
    else:
        df_previous = plays_df[plays_df['Season'] == (year - 1)].copy()
        print(f"  Using {year-1} season data for early weeks")
    
    # Add features
    df_with_features = create_team_tendency_features.add_team_tendency_features(
        df_season=df_season,
        df_previous_season=df_previous,
        global_means=global_means,
        team_col='TeamWithPossessionShort'
    )
    
    all_seasons_with_features.append(df_with_features)

# Combine all seasons back together
plays_df = pd.concat(all_seasons_with_features, ignore_index=True)

print(f"Added features to {len(plays_df):,} plays")
print(f"New columns: team_run_pct, team_short_pass_pct, team_long_pass_pct")


Processing 2017...
  Using global means (first season)

Processing 32 teams for season 2017...
Added team tendency features for 32 teams

Processing 2018...
  Using 2017 season data for early weeks

Processing 32 teams for season 2018...
Added team tendency features for 32 teams

Processing 2019...
  Using 2018 season data for early weeks

Processing 32 teams for season 2019...
Added team tendency features for 32 teams

Processing 2020...
  Using 2019 season data for early weeks

Processing 32 teams for season 2020...
Added team tendency features for 32 teams

Processing 2021...
  Using 2020 season data for early weeks

Processing 32 teams for season 2021...
Added team tendency features for 32 teams

Processing 2022...
  Using 2021 season data for early weeks

Processing 32 teams for season 2022...
Added team tendency features for 32 teams

Processing 2023...
  Using 2022 season data for early weeks

Processing 32 teams for season 2023...
Added team tendency features for 32 teams

Pro

In [23]:
print(plays_df.columns)

Index(['Season', 'Week', 'Day', 'Date', 'AwayTeam', 'HomeTeam', 'Quarter',
       'DriveNumber', 'TeamWithPossession', 'IsScoringDrive',
       'PlayNumberInDrive', 'IsScoringPlay', 'PlayOutcome', 'PlayStart',
       'PlayTimeFormation', 'PlayDescription', 'PlayCall',
       'TeamWithPossessionShort', 'Down', 'YdsTo1stDown', 'Territory',
       'YdPosition', 'YdsToEndzone', 'GameId', 'team_run_pct',
       'team_short_pass_pct', 'team_long_pass_pct'],
      dtype='object')


In [ ]:
# Add team tendency features for each season
all_seasons_with_features = []

for year in range(2017, 2026):  # 2017-2025
    print(f"\nProcessing {year}...")
    
    # Get current season data
    df_season = plays_df[plays_df['Season'] == year].copy()
    
    # Get previous season data (None for 2017)
    if year == 2017:
        df_previous = None
        print(f"  Using global means (first season)")
    else:
        df_previous = plays_df[plays_df['Season'] == (year - 1)].copy()
        print(f"  Using {year-1} season data for early weeks")
    
    # Add features
    df_with_features = create_team_tendency_features.add_team_tendency_features(
        df_season=df_season,
        df_previous_season=df_previous,
        global_means=global_means,
        team_col='TeamWithPossessionShort'
    )
    
    all_seasons_with_features.append(df_with_features)

# Combine all seasons back together
plays_df = pd.concat(all_seasons_with_features, ignore_index=True)

print(f"Added features to {len(plays_df):,} plays")
print(f"New columns: team_run_pct, team_short_pass_pct, team_long_pass_pct")


Processing 2017...
  Using global means (first season)

Processing 32 teams for season 2017...
Added team tendency features for 32 teams

Processing 2018...
  Using 2017 season data for early weeks

Processing 32 teams for season 2018...
Added team tendency features for 32 teams

Processing 2019...
  Using 2018 season data for early weeks

Processing 32 teams for season 2019...
Added team tendency features for 32 teams

Processing 2020...
  Using 2019 season data for early weeks

Processing 32 teams for season 2020...
Added team tendency features for 32 teams

Processing 2021...
  Using 2020 season data for early weeks

Processing 32 teams for season 2021...
Added team tendency features for 32 teams

Processing 2022...
  Using 2021 season data for early weeks

Processing 32 teams for season 2022...
Added team tendency features for 32 teams

Processing 2023...
  Using 2022 season data for early weeks

Processing 32 teams for season 2023...
Added team tendency features for 32 teams

Pro

In [25]:
clean_df = plays_df[["DriveNumber", "PlayCall", "Down", "YdsTo1stDown", "YdsToEndzone", "team_run_pct", "team_short_pass_pct", "team_long_pass_pct"]]
print(clean_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 285588 entries, 0 to 285587
Data columns (total 8 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   DriveNumber          285588 non-null  int64  
 1   PlayCall             285588 non-null  object 
 2   Down                 285588 non-null  int64  
 3   YdsTo1stDown         285588 non-null  int64  
 4   YdsToEndzone         285588 non-null  int64  
 5   team_run_pct         285588 non-null  float64
 6   team_short_pass_pct  285588 non-null  float64
 7   team_long_pass_pct   285588 non-null  float64
dtypes: float64(3), int64(4), object(1)
memory usage: 17.4+ MB
None


In [26]:
# save/create clean_df as a csv in our current directory, if it already exists then update it 
clean_df.to_csv('clean_plays_data.csv', index=False)
print("Data saved to clean_plays_data.csv")

Data saved to clean_plays_data.csv
